# 01. Россия в британской прессе: отбор ближайших соседей

**Вторая статья, первый ноутбук.** Три слова — `russia`, `russian`, `czar`; 12 десятилетий (1800–1919); Top-100 исходных кандидатов → проверка → Top-15. Не зависит от расчёта семантических осей.

**Порядок:** (1) извлечение из готового кэша; (2) применение зафиксированных OCR-решений и исключение прямых вариантов целевого слова; (3) чтение сохранённых решений ChatGPT; (4) применение подтверждённых автором исключений и контроль сомнительных записей; (5) построение Top-15; (6) аудит, сопоставление и проверки.

В ноутбуке **нет API-запросов**. Файл `chat_review.csv` содержит результаты двух раундов разметки GPT-6 от 22.11.2026. Утверждать, что эти ответы воспроизведены повторным запросом или что все слова проверены по газетным оригиналам, нельзя. Предположительные OCR-исключения отмечены как таковые.

Запускайте из корня репозитория. Рядом должны находиться `data/article2_neighbors/*.csv`. Если кэш `data/cache_2023/cache.pkl` найден, используются его исходные соседи с полной точностью. Иначе используется эквивалентный снимок первых ста соседей, включённый в комплект. **Никакой загрузки исходных моделей не требуется.**

## 0. Параметры и функции чтения

Чтобы изменить число соседей для публикации, достаточно параметров `N_CANDIDATES` и `N_RESULT`. Исходные слова и десятилетия фиксированы. Таблицы решений не переписываются кодом.

In [8]:
from pathlib import Path
from collections import Counter
import pickle
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
SOURCE = ROOT / "data" / "article2_neighbors"
OUT = ROOT / "results" / "article2_neighbors"
RAW_OUT = OUT / "raw_top100"
FOR_CHAT = OUT / "for_chat"
for folder in [OUT, RAW_OUT, FOR_CHAT]: folder.mkdir(parents=True, exist_ok=True)

TARGETS = ("russia", "russian", "czar")
DECADES = [f"{y}-е" for y in range(1800,1920,10)]
PERIODS = [f"{y}s" for y in range(1800,1920,10)]
N_CANDIDATES = 100
N_RESULT = 15
KEY = ["target","period","word"]
DECISIONS = {"keep", "exclude_ocr", "exclude_variant", "review"}

assert SOURCE.is_dir(), f"Не обнаружены входные CSV: {SOURCE}"

def load_csv(path):
    assert path.is_file(), f"Нет файла: {path}"
    return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")

def save_csv(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.17g")

def check_unique(data, columns, label):
    if data.duplicated(columns).any():
        raise ValueError(f"Повторные ключи в {label}: {data[data.duplicated(columns,keep=False)][columns].head()}")

print("Каталог проекта:", ROOT)
print("Top-100 → Top-15; периодов:", len(DECADES))

Каталог проекта: C:\Users\Admin\Documents\GitHub\british-press-semantic-change
Top-100 → Top-15; периодов: 12


## 1. Извлечение исходных 100 соседей

Если кэш имеется, берём `cache['neighbors']`, не перестраивая Word2Vec. Для повторяемости в комплект входят те же 100 строк на период, сформированные из кэша до нынешнего запуска. Ранг и косинус **не меняются** на всех последующих этапах.

In [9]:
cache_candidates = [ROOT / "data/cache_2023/cache.pkl", ROOT / "cache.pkl"]
cache_path = next((path for path in cache_candidates if path.is_file()), None)
if cache_path is not None:
    with cache_path.open("rb") as f:
        cache = pickle.load(f)  # только собственный доверенный кэш проекта
    raw_rows = []
    for target in TARGETS:
        for p, lab in zip(PERIODS, DECADES):
            neighbors = cache["neighbors"][p][target][:N_CANDIDATES]
            if len(neighbors) < N_CANDIDATES:
                raise ValueError(f"Недостаточно соседей в кэше: {target}, {p}")
            for rank, (word, cosine) in enumerate(neighbors, 1):
                raw_rows.append(dict(target=target, period=lab, raw_rank=rank,
                                     word=str(word).lower(), cosine=float(cosine)))
    raw = pd.DataFrame(raw_rows)
    source_name = str(cache_path)
else:
    raw = pd.concat([load_csv(SOURCE / f"{target}_source_top100.csv")
                     for target in TARGETS], ignore_index=True)
    source_name = "Фиксированные файлы source_top100 (снимок кэша)"
raw["raw_rank"] = raw["raw_rank"].astype(int)
raw["cosine"] = raw["cosine"].astype(float)
raw = raw.sort_values(["target","period","raw_rank"]).reset_index(drop=True)
check_unique(raw, KEY, "raw")
assert len(raw)==len(TARGETS)*len(DECADES)*N_CANDIDATES
for (target,period), g in raw.groupby(["target","period"]):
    assert list(g.raw_rank)==list(range(1,N_CANDIDATES+1)), (target,period)
    assert g.cosine.between(-1,1).all()
    assert g.cosine.is_monotonic_decreasing
for target in TARGETS:
    save_csv(raw[raw.target.eq(target)], RAW_OUT / f"{target}_raw_top100.csv")
print("Источники:",source_name)
print("Исходных кандидатов:",len(raw))
display(raw.head(6))

Источники: C:\Users\Admin\Documents\GitHub\british-press-semantic-change\data\cache_2023\cache.pkl
Исходных кандидатов: 3600


,target,period,raw_rank,word,cosine
0,czar,1800-е,1,alaric,0.636607
1,czar,1800-е,2,grofvenoes,0.635089
2,czar,1800-е,3,folcy,0.632369
3,czar,1800-е,4,mellifh,0.624929
4,czar,1800-е,5,skyscraper,0.623199
5,czar,1800-е,6,pacificator,0.622051


## 2. Автоматическое применение известных правил

Входной `prior_lexical_decisions.csv` — **снимок ранее принятых решений**, а не словарь, автоматически определяющий истинные ошибки. В нём есть исходная категория и её происхождение. Повторные написания целевого слова исключаются по **закрытому явному перечню** `target_variants.csv`, без нечёткого поиска по расстоянию Левенштейна. Отсутствие слова в словаре не является основанием для исключения.

Неразрешённые позиции перед предварительной границей пятнадцатого `keep` экспортируются в файлы `for_chat/`. При необходимости можно приложить такой CSV и текст промпта из файла `data/article2_neighbors/CHAT_PROMPT.md` в обычный чат.

In [10]:
prior = load_csv(SOURCE / "prior_lexical_decisions.csv")
check_unique(prior, KEY, "prior")
work = raw.merge(prior, on=KEY, how="left", validate="one_to_one")
assert work.legacy_decision.notna().all(), "Исходные решения не покрывают все 3600 строк"
work["decision"] = work.legacy_decision.replace({"exclude":"exclude_ocr"})
work["reason"] = work.legacy_reason
work["source"] = "прежняя лексическая разметка / " + work.legacy_source

variants = load_csv(SOURCE / "target_variants.csv")
check_unique(variants, ["target","word"], "target_variants")
for v in variants.itertuples(index=False):
    mask = work.target.eq(v.target) & work.word.eq(v.word)
    work.loc[mask,"decision"] = "exclude_variant"
    work.loc[mask,"reason"] = v.reason
    work.loc[mask,"source"] = "фиксированное правило варианта целевой лексемы"

# Сохраняем все нерешённые записи до предварительной границы + небольшой запас.
chat_input = []
for (target,period), g in work.groupby(["target","period"],sort=False):
    cutoff = int(g[g.decision.eq("keep")].iloc[N_RESULT-1].raw_rank)
    pending = g[g.decision.eq("review") & g.raw_rank.le(min(cutoff + 12,N_CANDIDATES))].copy()
    pending["preliminary_cutoff"] = cutoff
    chat_input.append(pending[["target","period","raw_rank","word","cosine",
                               "preliminary_cutoff","legacy_reason"]])
chat_input = pd.concat(chat_input, ignore_index=True)
for target in TARGETS:
    save_csv(chat_input[chat_input.target.eq(target)], FOR_CHAT / f"{target}_for_chat.csv")
print("Строк передано в очередь чата:",len(chat_input))
display(chat_input.groupby("target").size().rename("кандидатов").to_frame())

C:\Users\Admin\AppData\Local\Temp\ipykernel_23752\3794791115.py:26: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")
C:\Users\Admin\AppData\Local\Temp\ipykernel_23752\3794791115.py:26: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")


Строк передано в очередь чата: 252


,кандидатов
target,
russia,80
russian,172


## 3. Предварительная экспертиза GPT-6 в режиме чата

В этом запуске мы **не вызываем модель из кода**. `chat_review.csv` хранит решения, фактически полученные в чате 22.11.2026 (первоначальные 141 формы и повторный разбор оставшихся случаев). Исторические имена, иностранные формы и нарицательные слова не удаляются лишь потому, что они тематически неожиданны. При отсутствии данных для нового решения остаётся `review`.

Этот файл можно позднее обновить вручную, заменив строку и сохранив причину. Устаревшие исходные ранги не принимаются за основания для решения. Изменение статуса **не изменяет вектор** или косинус.

In [11]:
chat = load_csv(SOURCE / "chat_review.csv")
check_unique(chat, KEY, "chat_review")
assert set(chat.decision).issubset(DECISIONS)
assert chat.reason.astype(str).str.len().gt(5).all()
assert chat.reviewer.astype(str).str.len().gt(3).all()
assert set(map(tuple,chat[KEY].to_numpy())).issubset(set(map(tuple,work[KEY].to_numpy())))
chat_idx = chat.set_index(KEY)
work_idx = work.set_index(KEY)
for field, output in [("decision","decision"),("reason","reason")]:
    work_idx.loc[chat_idx.index,output] = chat_idx[field]
work_idx.loc[chat_idx.index,"source"] = "GPT-6, чат 22.11.2026 (" + chat_idx["decision_source"] + ")"
work = work_idx.reset_index().sort_values(["target","period","raw_rank"])
print("Размеченных ChatGPT строк:",len(chat))
display(chat.decision.value_counts().rename_axis("решение").to_frame("число"))

Размеченных ChatGPT строк: 158


C:\Users\Admin\AppData\Local\Temp\ipykernel_23752\3794791115.py:26: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")


,число
решение,
exclude_ocr,85
keep,64
review,9


## 4. Применение подтверждённых автором решений и контроль

`author_decisions.csv` содержит **только явно сформулированное автором проекта** решение об исключении в данном контексте `raffia`, `ruffle`, `ruffian`. Этот шаг имеет приоритет над предложениями модели. Иные решения ChatGPT являются **модельной экспертной разметкой**, а не независимой человеческой проверкой. Автор может дополнить этот файл после просмотра протокола.

In [12]:
author = load_csv(SOURCE / "author_decisions.csv")
check_unique(author, ["target","word"], "author_decisions")
assert set(author.decision).issubset(DECISIONS)
for r in author.itertuples(index=False):
    mask=work.target.eq(r.target)&work.word.eq(r.word)
    if not mask.any(): raise ValueError(f"Авторское правило отсутствует в данных: {r.target}:{r.word}")
    work.loc[mask,"decision"] = r.decision
    work.loc[mask,"reason"] = r.reason
    work.loc[mask,"source"] = r.reviewer
assert set(work.decision).issubset(DECISIONS)
assert not work.reason.isna().any()
print("Точечных авторских правил:",len(author))
print("Итоговая разметка 3600 кандидатов:")
display(work.decision.value_counts().rename_axis("статус").to_frame("записей"))

Точечных авторских правил: 3
Итоговая разметка 3600 кандидатов:


C:\Users\Admin\AppData\Local\Temp\ipykernel_23752\3794791115.py:26: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")


,записей
статус,
keep,1575
exclude_ocr,999
review,988
exclude_variant,38


## 5. Отбор пятнадцати ближайших допустимых соседей

Для каждого десятилетия сначала берём первые 15 слов с решением `keep`. Если перед последним из них осталась хотя бы одна строка `review`, **публикационный Top-15 не готов**. Неопределённость после этой границы не влияет на состав пятнадцатки и не требует срочного разрешения. Статусы `exclude_ocr` и `exclude_variant` не смешиваются в таблице оснований.

In [13]:
top_rows=[]; coverage_rows=[]; blocker_rows=[]
for (target,period), g in work.groupby(["target","period"],sort=False):
    g=g.sort_values("raw_rank")
    keep=g[g.decision.eq("keep")]
    if len(keep)<N_RESULT:
        raise ValueError(f"Недостаточно допустимых слов: {target}, {period}")
    cutoff=int(keep.iloc[N_RESULT-1].raw_rank)
    blockers=g[g.raw_rank.le(cutoff)&g.decision.eq("review")]
    counts=g.decision.value_counts()
    coverage_rows.append(dict(target=target, period=period, raw_rank_15=cutoff,
        blocking_reviews=len(blockers), ready=(len(blockers)==0),
        keep=int(counts.get("keep",0)), exclude_ocr=int(counts.get("exclude_ocr",0)),
        exclude_variant=int(counts.get("exclude_variant",0)), review=int(counts.get("review",0))))
    blocker_rows.append(blockers)
    for i,r in enumerate(keep.head(N_RESULT).itertuples(index=False),1):
        top_rows.append(dict(target=target,period=period,selected_rank=i,
                             raw_rank=int(r.raw_rank),word=r.word,cosine=float(r.cosine),
                             decision_source=r.source))
coverage=pd.DataFrame(coverage_rows)
blockers=pd.concat(blocker_rows,ignore_index=True)
if len(blockers):
    save_csv(blockers,OUT/"blocking_reviews.csv")
    raise ValueError(f"Остались {len(blockers)} нерешённых строк перед Top-15: откройте blocking_reviews.csv")
top=pd.DataFrame(top_rows)
assert len(top)==len(TARGETS)*len(DECADES)*N_RESULT
save_csv(top,OUT/"top15_long.csv")
for target in TARGETS:
    t=top[top.target.eq(target)]
    wide=(t.assign(display=t.word+" ("+t.cosine.map(lambda x:f"{x:.3f}")+")")
             .pivot(index="selected_rank", columns="period", values="display")
             .reindex(index=range(1,N_RESULT+1),columns=DECADES))
    save_csv(wide.reset_index().rename(columns={"selected_rank":"rank"}),OUT/f"{target}_top15_wide.csv")
save_csv(coverage,OUT/"coverage.csv")
print(f"Top-15 сформированы: {len(top)} записей; все {len(coverage)} списка без блокирующих REVIEW")
print("Максимальная исходная позиция 15-го соседа:",coverage.raw_rank_15.max())
display(coverage.groupby("target").agg(max_raw_rank15=("raw_rank_15","max"),
    min_raw_rank15=("raw_rank_15","min")).round(0))

Top-15 сформированы: 540 записей; все 36 списка без блокирующих REVIEW
Максимальная исходная позиция 15-го соседа: 84


,max_raw_rank15,min_raw_rank15
target,,
czar,47,15
russia,84,15
russian,52,16


## 6. Аудит и проверка чувствительности

Сохраняются полный протокол 3600 кандидатов, три таблицы Top-15, список неразрешённых форм **ниже границы отбора**, сравнение с таблицей *russia* прежней версии статьи и контрольный сценарий сохранения спорных реальных слов, исключённых как предполагаемые OCR-повреждения. Это **проверка влияния решения**, не статистический доверительный интервал.

In [14]:
save_csv(work[["target","period","raw_rank","word","cosine","decision","source","reason"]],
         OUT/"full_decision_audit_top100.csv")
remaining=[]
for (target,period),g in work.groupby(["target","period"],sort=False):
    cutoff=int(coverage[(coverage.target.eq(target))&(coverage.period.eq(period))].iloc[0].raw_rank_15)
    remaining.append(g[g.decision.eq("review")&g.raw_rank.gt(cutoff)])
remaining=pd.concat(remaining,ignore_index=True)
save_csv(remaining[["target","period","raw_rank","word","cosine","reason","source"]],
         OUT/"nonblocking_review.csv")

# Проверяем влияние исключения вероятных OCR-артефактов, которые могут быть реальными словами.
BORDERLINE={"russia":{"raffia","ruffle","ruffs","rulla","runia","prance"},
            "russian":{"ruffian","sian","russiam","garian"},"czar":set()}
sensitivity=[]
for (target,period),g in work.groupby(["target","period"],sort=False):
    original=top[(top.target.eq(target))&(top.period.eq(period))].word.tolist()
    g=g.sort_values("raw_rank")
    relaxed=g[g.decision.eq("keep")|((g.word.isin(BORDERLINE[target]))&g.decision.eq("exclude_ocr"))]
    alt=relaxed.head(N_RESULT).word.tolist()
    sensitivity.append(dict(target=target,period=period,
      unchanged_positions=sum(a==b for a,b in zip(original,alt)),
      overlap=len(set(original)&set(alt)),
      excluded_borderline_words="; ".join(g[g.word.isin(BORDERLINE[target])&g.decision.eq("exclude_ocr")].word.tolist()),
      main_first15="; ".join(original), alternative_first15="; ".join(alt)))
sensitivity=pd.DataFrame(sensitivity)
save_csv(sensitivity, OUT/"sensitivity_borderline_ocr.csv")

reference=load_csv(SOURCE/"reference_article26_russia_top10.csv")
rows=[]
for p in DECADES:
    old=reference[reference.period.eq(p)].sort_values("rank").word.tolist()
    new=top[(top.target.eq("russia"))&top.period.eq(p)].sort_values("selected_rank").word.head(10).tolist()
    rows.append(dict(period=p,old_top10="; ".join(old),new_top10="; ".join(new),
        exact_positions_equal=sum(a==b for a,b in zip(old,new)),
        set_overlap=len(set(old)&set(new)),
        words_entered="; ".join(x for x in new if x not in old),
        words_removed="; ".join(x for x in old if x not in new)))
comparison=pd.DataFrame(rows)
save_csv(comparison,OUT/"comparison_article26_russia_top10.csv")

# Аудит: никакие ранги и косинусы не менялись; Top-15 — подмножество исходных кандидатов.
chk=top.merge(raw,on=KEY,validate="one_to_one",suffixes=("_selected","_raw"))
assert (chk.raw_rank_selected==chk.raw_rank_raw).all()
assert np.allclose(chk.cosine_selected,chk.cosine_raw,rtol=0,atol=1e-14)
assert coverage.ready.all()
assert coverage.raw_rank_15.max()<=N_CANDIDATES
print("Все проверки пройдены.")
print("Совпадение с прежним Top-10 Russia по десятилетиям:")
display(comparison[["period","exact_positions_equal","set_overlap","words_entered","words_removed"]])
print("Число списков, чувствительных к сохранению спорных OCR-гомографов:",
      sum(sensitivity.overlap.lt(N_RESULT)))

Все проверки пройдены.
Совпадение с прежним Top-10 Russia по десятилетиям:


C:\Users\Admin\AppData\Local\Temp\ipykernel_23752\3794791115.py:26: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, float_precision="round_trip")


,period,exact_positions_equal,set_overlap,words_entered,words_removed
0,1800-е,5,8,kuala; rus,continental; mediation
1,1810-е,4,9,rossi,russian
2,1820-е,7,9,ruda,moldavia
3,1830-е,10,10,,
4,1840-е,10,10,,
5,1850-е,10,10,,
6,1860-е,10,10,,
7,1870-е,10,10,,
8,1880-е,10,10,,
9,1890-е,10,10,,


Число списков, чувствительных к сохранению спорных OCR-гомографов: 8


## Методологические оговорки и воспроизведение

- Кандидаты извлекаются по косинусу. На этапе фильтрации векторы и исходное соседство не пересчитываются.
- `exclude_variant` означает повторное обозначение целевого слова, **не обязательно OCR-ошибку**; `exclude_ocr` включает вероятные, но не гарантированные ошибки распознавания.
- Решения GPT-6 в `chat_review.csv` относятся к лексической форме и её вероятному происхождению. Исторический контекст газетного употребления **для всех записей не проверен**, и в архиве не имеется его полного текста.
- Существующие английские слова (`raffia`, `ruffle`, `ruffian`) исключены по прямо высказанному автором предположению о вероятном OCR-смешении; альтернативный сценарий в `sensitivity_borderline_ocr.csv` показывает зависимость списков от этого выбора.
- Следует отличать *модельную экспертную проверку* от независимого подтверждения автором. В этом комплекте автор явно подтвердил только указанные правила; остальные решения модели доступны для просмотра и при необходимости изменения до публикации.
- Это отдельный ноутбук для второй статьи. Семантические оси не изменяются. Файлы первой статьи не перезаписываются.